# Lesson 04 — Warping One Image into Another

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M    = cv2.getRotationMatrix2D((img1.shape[1]//2,img1.shape[0]//2),25,0.8)
img2 = cv2.warpAffine(img2, M, (img2.shape[1],img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=500)
kp1,d1 = sift.detectAndCompute(cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY),None)
kp2,d2 = sift.detectAndCompute(cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY),None)
good   = [m for m,n in cv2.BFMatcher().knnMatch(d1,d2,k=2) if m.distance<0.75*n.distance]

src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)

# Warp img1 into img2's coordinate system
h2, w2 = img2.shape[:2]
warped  = cv2.warpPerspective(img1, H, (w2, h2))

# Alpha blend the two
blend = cv2.addWeighted(img2, 0.5, warped, 0.5, 0)

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].imshow(cv2.cvtColor(img2,  cv2.COLOR_BGR2RGB)); axes[0].set_title('Image 2'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(warped,cv2.COLOR_BGR2RGB)); axes[1].set_title('Image 1 warped to Image 2'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB)); axes[2].set_title('50% blend — alignment check'); axes[2].axis('off')
plt.show()

## Key Takeaway
Good alignment = the 50% blend looks like one seamless image.
Misalignment shows as ghosting/doubling. Use this blend trick to verify your homography.